# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [18]:
import os
import certifi
import duckdb
from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()


get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir

In [11]:
def get_sql_df(sql_query):
    con = get_duckdb_connection()
    
    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)
        
        if schema == "bronze":
            return f"{keyword} read_parquet('{DATA_DIR}/{schema}/{table}/**/*.parquet')"
        else:
            return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)', 
        replace_table, 
        sql_query
    )
    return con.execute(parsed_query).df()

def print_sql_df(sql_query):
    df = get_sql_df(sql_query)
    display(df)
    return df

In [3]:
print("Bronze Daily Flows Summary:")
print_sql_df('select * from bronze.electricity_flows order by process_ts desc limit 10')

Bronze Daily Flows Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777136260728,2026-04-25 22:27:40.728139+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-25 20:30:00+05:30,2026-04-25 21:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",1,25,04,2026
1,1777129595150,2026-04-25 20:36:35.150267+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-24 20:30:00+05:30,2026-04-25 20:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [4]:
print("Bronze Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix order by process_ts desc limit 10')

Bronze Daily Mix Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777136260728,2026-04-25 22:27:40.728139+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-25 20:30:00+05:30,2026-04-25 21:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",1,25,04,2026
1,1777129595150,2026-04-25 20:36:35.150267+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-24 20:30:00+05:30,2026-04-25 20:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [5]:
print("Silver :")
print_sql_df('select * from silver.electricity_mix order by process_ts desc limit 10')

Silver :


,process_ts,zone,datetime,updated_at,is_estimated,estimation_method,nuclear_mw,geothermal_mw,biomass_mw,coal_mw,...,unknown_mw,hydro_storage_charge_mw,hydro_storage_discharge_mw,battery_storage_charge_mw,battery_storage_discharge_mw,flow_exports_mw,flow_imports_mw,year,month,day
0,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,True,SANDBOX_MODE_DATA,24702.981,NaN,928.574,0.0,...,NaN,1616.533,NaN,11.012,NaN,8199.0,214.0,2026,4,25
1,1777129719141,FR,2026-04-24 20:30:00+05:30,2026-04-24 22:40:58.002000+05:30,True,SANDBOX_MODE_DATA,27602.993,NaN,913.542,0.0,...,NaN,1130.141,NaN,3.146,NaN,18556.0,0.0,2026,4,24
2,1777129719141,FR,2026-04-24 21:30:00+05:30,2026-04-24 23:42:55.823000+05:30,True,SANDBOX_MODE_DATA,44096.091,NaN,1414.883,0.0,...,NaN,219.090,NaN,3.936,NaN,15403.0,0.0,2026,4,24
3,1777129719141,FR,2026-04-24 22:30:00+05:30,2026-04-25 00:25:00.889000+05:30,True,SANDBOX_MODE_DATA,37933.075,NaN,1060.536,0.0,...,NaN,NaN,2044.104,NaN,3.403,15551.0,0.0,2026,4,24
4,1777129719141,FR,2026-04-24 23:30:00+05:30,2026-04-25 01:40:10.039000+05:30,True,SANDBOX_MODE_DATA,38634.294,NaN,1112.490,0.0,...,NaN,NaN,4150.292,NaN,124.230,20794.0,0.0,2026,4,24
5,1777129719141,FR,2026-04-25 00:30:00+05:30,2026-04-25 02:54:54.199000+05:30,True,SANDBOX_MODE_DATA,52236.370,NaN,1052.746,0.0,...,NaN,NaN,3013.633,NaN,132.386,18232.0,0.0,2026,4,24
6,1777129719141,FR,2026-04-25 01:30:00+05:30,2026-04-25 03:40:05.174000+05:30,True,SANDBOX_MODE_DATA,48019.536,NaN,1360.869,0.0,...,NaN,NaN,2959.524,16.907,NaN,16826.0,0.0,2026,4,24
7,1777129719141,FR,2026-04-25 02:30:00+05:30,2026-04-25 04:54:57.101000+05:30,True,SANDBOX_MODE_DATA,52953.391,NaN,1260.479,0.0,...,NaN,NaN,3508.791,1.377,NaN,15847.0,0.0,2026,4,24
8,1777129719141,FR,2026-04-25 03:30:00+05:30,2026-04-25 06:10:30.076000+05:30,True,SANDBOX_MODE_DATA,44614.105,NaN,1285.635,0.0,...,NaN,NaN,3974.419,NaN,3.536,20380.0,0.0,2026,4,24
9,1777129719141,FR,2026-04-25 04:30:00+05:30,2026-04-25 06:40:20.284000+05:30,True,SANDBOX_MODE_DATA,41766.713,NaN,1294.023,0.0,...,NaN,NaN,3025.192,NaN,19.199,17052.0,0.0,2026,4,24


In [6]:
print("Silver :")
print_sql_df('select * from silver.electricity_flows order by process_ts desc limit 10')

Silver :


,process_ts,zone,datetime,updated_at,neighbor_zone,direction,power_mw,year,month,day
0,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,BE,import,1284.0,2026,4,25
1,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,ES,import,1858.0,2026,4,25
2,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,CH,export,191.0,2026,4,25
3,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,DE,import,276.0,2026,4,25
4,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,IT-NO,export,564.0,2026,4,25
5,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,GB,export,3016.0,2026,4,25
6,1777136296129,FR,2026-04-25 20:30:00+05:30,2026-04-25 22:11:13.088000+05:30,LU,export,185.0,2026,4,25
7,1777129719141,FR,2026-04-24 20:30:00+05:30,2026-04-24 22:40:58.002000+05:30,ES,import,2543.0,2026,4,24
8,1777129719141,FR,2026-04-24 21:30:00+05:30,2026-04-24 23:42:55.823000+05:30,CH,import,2411.0,2026,4,24
9,1777129719141,FR,2026-04-24 21:30:00+05:30,2026-04-24 23:42:55.823000+05:30,BE,import,2395.0,2026,4,24


In [7]:
print("Gold Daily Exports:")
print_sql_df('select * from gold.daily_exports order by process_ts desc limit 10')

Gold Daily Exports:


,process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month,day
0,1777136466868,FR,France,LU,LU,2026-04-25,2945.0,16,2026,4,25
1,1777136466868,FR,France,GB,Great Britain,2026-04-25,46908.0,16,2026,4,25
2,1777136466868,FR,France,IT-NO,Italy North,2026-04-25,17112.0,16,2026,4,25
3,1777129788867,FR,France,GB,Great Britain,2026-04-24,27489.0,9,2026,4,24
4,1777129788867,FR,France,LU,LU,2026-04-24,1531.0,9,2026,4,24
5,1777129788867,FR,France,IT-NO,Italy North,2026-04-24,33987.0,9,2026,4,24
6,1777129788867,FR,France,IT-NO,Italy North,2026-04-25,16548.0,15,2026,4,25
7,1777129788867,FR,France,GB,Great Britain,2026-04-25,43892.0,15,2026,4,25
8,1777129788867,FR,France,LU,LU,2026-04-25,2760.0,15,2026,4,25


In [8]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports order by process_ts desc LIMIT 10")


Gold Daily Imports:


,process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month,day
0,1777136466868,FR,France,DE,Germany,2026-04-25,12461.0,16,2026,4,25
1,1777136466868,FR,France,CH,Switzerland,2026-04-25,2930.0,16,2026,4,25
2,1777136466868,FR,France,BE,Belgium,2026-04-25,39728.0,16,2026,4,25
3,1777136466868,FR,France,ES,Spain,2026-04-25,42045.0,16,2026,4,25
4,1777129788867,FR,France,DE,Germany,2026-04-25,12185.0,15,2026,4,25
5,1777129788867,FR,France,BE,Belgium,2026-04-25,38444.0,15,2026,4,25
6,1777129788867,FR,France,ES,Spain,2026-04-25,40187.0,15,2026,4,25
7,1777129788867,FR,France,BE,Belgium,2026-04-24,33768.0,9,2026,4,24
8,1777129788867,FR,France,DE,Germany,2026-04-24,21702.0,9,2026,4,24
9,1777129788867,FR,France,CH,Switzerland,2026-04-25,3121.0,15,2026,4,25


In [9]:
print("\nGold Daily Mix:")
print_sql_df("SELECT * FROM gold.daily_electricity_mix order by process_ts desc LIMIT 10")


Gold Daily Mix:


,process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,...,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month,day
0,1777136466868,FR,France,2026-04-25,73.935531,2.202645,3.139444,11.534831,8.443589,0.672814,...,0.0,0.0,0.0,847009.645,99.256040,25.320509,16,2026,4,25
1,1777129788867,FR,France,2026-04-24,73.854184,2.047965,8.502293,4.398373,10.493044,0.642080,...,0.0,0.0,0.0,525165.332,99.295859,25.441676,9,2026,4,24
2,1777129788867,FR,France,2026-04-25,74.282383,2.189190,3.195997,11.189395,8.409479,0.663955,...,0.0,0.0,0.0,809799.140,99.266444,24.984061,15,2026,4,25


In [10]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")


Pipeline State (el_state):


,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777136466868,2026-04-25 22:31:13.454732+05:30,2026-04-25 22:31:19.401462+05:30,R,8,None
1,silver,1777136296129,2026-04-25 22:28:17.843746+05:30,2026-04-25 22:28:22.954757+05:30,C,8,None
2,bronze,1777136260728,2026-04-25 22:27:40.728139+05:30,2026-04-25 22:27:55.419097+05:30,C,2,None
3,gold,1777129788867,2026-04-25 20:39:50.371386+05:30,2026-04-25 20:39:55.353971+05:30,R,16,None
4,silver,1777129719141,2026-04-25 20:38:45.969940+05:30,2026-04-25 20:38:51.653279+05:30,C,197,None
5,bronze,1777129595150,2026-04-25 20:36:35.150267+05:30,2026-04-25 20:36:44.086418+05:30,C,48,None
